In [1]:
# =============================================================================
# CELL 1 — Imports and environment verification
#
# Kernel: CFB Model (ARM)  ~/miniforge3/envs/cfb_model_arm/bin/python
# NumPyro 0.21.0 / JAX 0.10.0 confirmed working on cpu backend (Day 20).
#
# DB connection opened here and held open for the full notebook.
# Do not close until Cell 9.
#
# Output: int.int_game_model_features — one row per team per game,
# all engineered features required by model_cfb(), written once and
# read by every subsequent model notebook.
# =============================================================================

import numpy as np
import pandas as pd
import psycopg2
import json
import time
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

DEV_SEASONS   = [2022, 2023, 2024]
HOLDOUT_SEASON = 2025
ARTIFACT_DIR  = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

conn = psycopg2.connect(
    host='127.0.0.1', port=5455, dbname='postgres',
    user='postgres', password='postgres'
)
cur = conn.cursor()

print(f"psycopg2     : {psycopg2.__version__}")
print(f"pandas       : {pd.__version__}")
print(f"numpy        : {np.__version__}")
print(f"sklearn      : {__import__('sklearn').__version__}")
print(f"\nDev seasons  : {DEV_SEASONS}")
print(f"Holdout      : {HOLDOUT_SEASON}")
print("\nDB connection established.")
print("Cell 1 complete.")

psycopg2     : 2.9.12 (dt dec pq3 ext lo64)
pandas       : 3.0.2
numpy        : 2.4.4
sklearn      : 1.8.0

Dev seasons  : [2022, 2023, 2024]
Holdout      : 2025

DB connection established.
Cell 1 complete.


In [2]:
# =============================================================================
# CELL 2 — Build valid FBS conference game pool (temp table)
#
# Authoritative source for conference: int_team_season_features.
# Both home and away teams must have a conference row that is not
# 'FBS Independents'. INNER JOIN handles non-FBS teams (no season row).
#
# Season filter: IN (2022, 2023, 2024). 2025 is holdout — never touched.
#
# Output: temp_valid_fbs_games — one row per game, used as the join
# anchor for all subsequent feature engineering cells.
# =============================================================================

valid_games_sql = """
    DROP TABLE IF EXISTS temp_valid_fbs_games;

    CREATE TEMP TABLE temp_valid_fbs_games AS
    SELECT
        g.id          AS game_id,
        g.season,
        g.week,
        g.home_team,
        g.away_team,
        g.neutral_site,
        g.home_points,
        g.away_points,
        home_sf.conference AS home_conference,
        away_sf.conference AS away_conference,
        home_sf.sp_rating  AS home_sp_rating,
        away_sf.sp_rating  AS away_sp_rating
    FROM raw.games g
    INNER JOIN int.int_team_season_features home_sf
        ON home_sf.team_name = g.home_team
       AND home_sf.season    = g.season
    INNER JOIN int.int_team_season_features away_sf
        ON away_sf.team_name = g.away_team
       AND away_sf.season    = g.season
    WHERE g.season IN (2022, 2023, 2024)
      AND g.conference_game = TRUE
      AND home_sf.conference != 'FBS Independents'
      AND away_sf.conference != 'FBS Independents';

    ALTER TABLE temp_valid_fbs_games
    ADD PRIMARY KEY (game_id);
"""

cur.execute(valid_games_sql)
conn.commit()

cur.execute("SELECT * FROM temp_valid_fbs_games ORDER BY season, week, game_id")
rows = cur.fetchall()
cols = [d[0] for d in cur.description]
valid_games = pd.DataFrame(rows, columns=cols)

numeric_cols = ['home_points', 'away_points', 'home_sp_rating', 'away_sp_rating']
valid_games[numeric_cols] = valid_games[numeric_cols].astype(float)

# --- FBS integrity check ---
print("Home conference distribution:")
print(valid_games['home_conference'].value_counts().sort_index().to_string())
print()
print("Away conference distribution:")
print(valid_games['away_conference'].value_counts().sort_index().to_string())
print()

assert 'FBS Independents' not in valid_games['home_conference'].values, \
    "FBS Independents found in home_conference — fix before proceeding"
assert 'FBS Independents' not in valid_games['away_conference'].values, \
    "FBS Independents found in away_conference — fix before proceeding"
assert HOLDOUT_SEASON not in valid_games['season'].values, \
    "2025 holdout leaked into valid_games — fix before proceeding"
assert valid_games['game_id'].isna().sum() == 0, \
    "Null game_id found"

print(f"Valid FBS conference games : {len(valid_games):,}")
print(f"Seasons                   : {sorted(valid_games['season'].unique())}")
print()
print("Cell 2 complete.")

Home conference distribution:
home_conference
ACC                  182
American Athletic    160
Big 12               183
Big Ten              210
Conference USA       123
Mid-American         147
Mountain West        141
Pac-12               111
SEC                  179
Sun Belt             171

Away conference distribution:
away_conference
ACC                  182
American Athletic    160
Big 12               183
Big Ten              210
Conference USA       123
Mid-American         147
Mountain West        141
Pac-12               111
SEC                  179
Sun Belt             171

Valid FBS conference games : 1,607
Seasons                   : [np.int64(2022), np.int64(2023), np.int64(2024)]

Cell 2 complete.


In [3]:
# =============================================================================
# CELL 3 — Compute close_game_play_count_delta
#
# Source: int_game_team_features — two rows per game (one per team).
# close_game_play_count     : plays run by this team in close-game situations
# close_game_def_play_count : plays faced by this team in close-game situations
#
# Delta = team's own close-game plays minus opponent's close-game plays faced.
# Positive = team ran more plays in close situations than opponent.
#
# Joined to temp_valid_fbs_games to restrict to training game pool only.
# =============================================================================

close_play_sql = """
    SELECT
        f.game_id,
        f.season,
        f.team_name,
        f.close_game_play_count,
        f.close_game_def_play_count
    FROM int.int_game_team_features f
    INNER JOIN temp_valid_fbs_games vg
        ON vg.game_id = f.game_id
    WHERE f.season IN (2022, 2023, 2024)
"""

cur.execute(close_play_sql)
rows = cur.fetchall()
cols = [d[0] for d in cur.description]
df_close = pd.DataFrame(rows, columns=cols)

numeric_cols = ['close_game_play_count', 'close_game_def_play_count']
df_close[numeric_cols] = df_close[numeric_cols].astype(float)

df_close['close_game_play_count_delta'] = (
    df_close['close_game_play_count'] - df_close['close_game_def_play_count']
)

print(f"Rows loaded          : {len(df_close):,}")
print(f"Expected (1607 × 2)  : {1607 * 2:,}")
print(f"Null delta           : {df_close['close_game_play_count_delta'].isna().sum()}")
print(f"Mean delta           : {df_close['close_game_play_count_delta'].mean():.3f}")
print(f"Std delta            : {df_close['close_game_play_count_delta'].std():.3f}")

assert len(df_close) == len(valid_games) * 2, \
    f"Row count mismatch: {len(df_close)} != {len(valid_games) * 2}"
assert df_close['close_game_play_count_delta'].isna().sum() == 0, \
    "Unexpected nulls in close_game_play_count_delta"

print()
print("Cell 3 complete.")

Rows loaded          : 3,214
Expected (1607 × 2)  : 3,214
Null delta           : 0
Mean delta           : 0.000
Std delta            : 15.817

Cell 3 complete.


In [4]:
# =============================================================================
# CELL 4 — Join int_game_environment and compute wind_chill
#
# Source: int_game_environment — one row per game (home_team/away_team,
# not team_name). Join on game_id only, then expand to two team rows
# by stacking home and away.
#
# wind_chill formula: NWS standard (same as EDA 06)
#   Active when: NOT is_dome AND T < 50F AND W > 3 mph
#   When conditions not met: wind_chill = temperature_f (no cold effect)
#   When is_dome or missing T/W: NaN -> zeroed in model prep (Cell 7)
#
# Threshold for model activation: wind_chill <= 40F AND NOT is_dome
# Pre-zeroing at threshold happens in Cell 7, not here.
# Here we compute the raw wind_chill value only.
# =============================================================================

env_sql = """
    SELECT
        e.game_id,
        e.home_team,
        e.away_team,
        e.is_dome,
        e.temperature_f,
        e.wind_speed_mph,
        e.away_elevation_delta_ft,
        e.away_travel_distance_mi,
        e.away_tz_delta_hrs
    FROM int.int_game_environment e
    INNER JOIN temp_valid_fbs_games vg
        ON vg.game_id = e.game_id
"""

cur.execute(env_sql)
rows = cur.fetchall()
cols = [d[0] for d in cur.description]
df_env = pd.DataFrame(rows, columns=cols)

# Cast boolean
df_env['is_dome'] = df_env['is_dome'].map(
    lambda x: 1 if x is True else (0 if x is False else np.nan)
).astype(float)

# Cast numeric
env_numeric = [
    'temperature_f', 'wind_speed_mph',
    'away_elevation_delta_ft', 'away_travel_distance_mi', 'away_tz_delta_hrs'
]
df_env[env_numeric] = df_env[env_numeric].astype(float)

# --- Compute wind_chill (exact EDA 06 formula) ---
def compute_wind_chill(row):
    T = row['temperature_f']
    W = row['wind_speed_mph']
    if row['is_dome'] == 1:
        return np.nan
    if pd.isna(T) or pd.isna(W):
        return np.nan
    if T >= 50 or W <= 3:
        return T  # conditions not met, wind chill = temp
    return (35.74 + 0.6215 * T - 35.75 * (W ** 0.16) + 0.4275 * T * (W ** 0.16))

df_env['wind_chill'] = df_env.apply(compute_wind_chill, axis=1)

# --- Expand to two team rows per game ---
# Home team row: away_elevation_delta_ft is from away team's perspective —
# irrelevant for home team row; will be zeroed by threshold in Cell 7.
# Away team row: carries the actual elevation/travel/tz values.
home_rows = df_env[['game_id', 'home_team', 'wind_chill', 'is_dome']].copy()
home_rows = home_rows.rename(columns={'home_team': 'team_name'})
home_rows['away_elevation_delta_ft'] = 0.0
home_rows['away_travel_distance_mi'] = 0.0
home_rows['away_tz_delta_hrs']       = 0.0

away_rows = df_env[[
    'game_id', 'away_team', 'wind_chill', 'is_dome',
    'away_elevation_delta_ft', 'away_travel_distance_mi', 'away_tz_delta_hrs'
]].copy()
away_rows = away_rows.rename(columns={'away_team': 'team_name'})

df_env_long = pd.concat([home_rows, away_rows], ignore_index=True)

print(f"Games in env table   : {len(df_env):,}")
print(f"Rows after expansion : {len(df_env_long):,}")
print(f"Expected             : {len(valid_games) * 2:,}")
print()
print(f"wind_chill non-null  : {df_env_long['wind_chill'].notna().sum():,}")
print(f"wind_chill null      : {df_env_long['wind_chill'].isna().sum():,}")
print(f"wind_chill <= 40F    : {(df_env_long['wind_chill'] <= 40).sum():,}")
print(f"wind_chill mean      : {df_env_long['wind_chill'].dropna().mean():.1f}F")
print()
print(f"away_elevation non-zero (away rows): {(df_env_long['away_elevation_delta_ft'] != 0).sum():,}")
print(f"away_travel non-zero  (away rows)  : {(df_env_long['away_travel_distance_mi'] != 0).sum():,}")
print(f"away_tz non-zero      (away rows)  : {(df_env_long['away_tz_delta_hrs'] != 0).sum():,}")

assert len(df_env_long) == len(valid_games) * 2, \
    f"Row count mismatch: {len(df_env_long)} != {len(valid_games) * 2}"

print()
print("Cell 4 complete.")

Games in env table   : 1,607
Rows after expansion : 3,214
Expected             : 3,214

wind_chill non-null  : 3,088
wind_chill null      : 126
wind_chill <= 40F    : 344
wind_chill mean      : 62.2F

away_elevation non-zero (away rows): 1,601
away_travel non-zero  (away rows)  : 1,607
away_tz non-zero      (away rows)  : 615

Cell 4 complete.


In [5]:
# =============================================================================
# CELL 5 — Aggregate raw.plays into per-team per-game style metrics
#
# Computes rush_rate_std_downs, rush_rate_pass_downs, off_sack_rate_allowed,
# and def_sack_rate per team per game, then derives home-minus-away deltas.
#
# Exact SQL from EDA 09. Play type definitions and down/distance filters
# match session state schema facts exactly.
#
# CRITICAL PERFORMANCE: valid game_ids materialized into a temp table with
# PRIMARY KEY before joining raw.plays — never scan with WHERE game_id IN
# (large list) or multiple INNER JOINs on raw.plays directly.
#
# Output: df_style — one row per team per game with four delta columns.
# =============================================================================

start_time = time.time()

style_sql = """
    DROP TABLE IF EXISTS temp_style_metrics;

    CREATE TEMP TABLE temp_style_metrics AS
    WITH scrimmage_plays AS (
        SELECT
            p.game_id,
            p.season,
            p.offense AS team_name,
            p.defense AS opponent,
            p.play_type,
            p.down,
            p.distance,

            CASE
                WHEN p.play_type IN ('Rush', 'Rushing Touchdown') THEN 1
                ELSE 0
            END AS is_rush,

            CASE
                WHEN p.play_type IN (
                    'Pass Reception',
                    'Pass Incompletion',
                    'Passing Touchdown',
                    'Sack',
                    'Pass Completion',
                    'Pass Interception Return'
                ) THEN 1
                ELSE 0
            END AS is_pass,

            CASE
                WHEN p.down = 1
                  OR (p.down = 2 AND p.distance <= 8)
                  OR (p.down IN (3, 4) AND p.distance <= 5)
                THEN 1
                ELSE 0
            END AS is_std_down,

            CASE
                WHEN (p.down = 2 AND p.distance > 8)
                  OR (p.down IN (3, 4) AND p.distance > 5)
                THEN 1
                ELSE 0
            END AS is_pass_down,

            CASE
                WHEN p.play_type = 'Sack' THEN 1
                ELSE 0
            END AS is_sack

        FROM raw.plays p
        INNER JOIN temp_valid_fbs_games vg
            ON vg.game_id = p.game_id
        WHERE p.season IN (2022, 2023, 2024)
    ),

    offense_metrics AS (
        SELECT
            game_id,
            season,
            team_name,
            opponent,
            AVG(CASE WHEN is_std_down  = 1 THEN is_rush::numeric END) AS rush_rate_std_downs,
            AVG(CASE WHEN is_pass_down = 1 THEN is_rush::numeric END) AS rush_rate_pass_downs,
            AVG(CASE WHEN is_pass      = 1 THEN is_sack::numeric END) AS off_sack_rate_allowed
        FROM scrimmage_plays
        GROUP BY game_id, season, team_name, opponent
    ),

    defense_metrics AS (
        SELECT
            game_id,
            opponent AS team_name,
            AVG(CASE WHEN is_pass = 1 THEN is_sack::numeric END) AS def_sack_rate
        FROM scrimmage_plays
        GROUP BY game_id, opponent
    )

    SELECT
        o.game_id,
        o.season,
        o.team_name,
        o.opponent,
        o.rush_rate_std_downs,
        o.rush_rate_pass_downs,
        o.off_sack_rate_allowed,
        d.def_sack_rate
    FROM offense_metrics o
    INNER JOIN defense_metrics d
        ON d.game_id   = o.game_id
       AND d.team_name = o.team_name;

    ALTER TABLE temp_style_metrics
    ADD PRIMARY KEY (game_id, team_name);
"""

cur.execute(style_sql)
conn.commit()

elapsed = time.time() - start_time

cur.execute("""
    SELECT * FROM temp_style_metrics
    ORDER BY season, game_id, team_name
""")
rows = cur.fetchall()
cols = [d[0] for d in cur.description]
df_style_raw = pd.DataFrame(rows, columns=cols)

style_numeric = [
    'rush_rate_std_downs', 'rush_rate_pass_downs',
    'off_sack_rate_allowed', 'def_sack_rate'
]
df_style_raw[style_numeric] = df_style_raw[style_numeric].astype(float)

# --- Compute home-minus-away deltas ---
# Join to valid_games to identify home/away orientation per game_id
home_style = df_style_raw.merge(
    valid_games[['game_id', 'home_team', 'away_team']],
    on='game_id', how='inner'
)

home_metrics = home_style[
    home_style['team_name'] == home_style['home_team']
][['game_id'] + style_numeric].rename(
    columns={c: f'home_{c}' for c in style_numeric}
)

away_metrics = home_style[
    home_style['team_name'] == home_style['away_team']
][['game_id'] + style_numeric].rename(
    columns={c: f'away_{c}' for c in style_numeric}
)

style_delta = home_metrics.merge(away_metrics, on='game_id', how='inner')

for col in style_numeric:
    style_delta[f'{col}_delta'] = (
        style_delta[f'home_{col}'] - style_delta[f'away_{col}']
    )

# Expand back to two team rows per game
delta_cols = [f'{c}_delta' for c in style_numeric]

home_delta = style_delta[['game_id'] + delta_cols].copy()
home_delta = home_delta.merge(
    valid_games[['game_id', 'home_team']].rename(columns={'home_team': 'team_name'}),
    on='game_id', how='inner'
)

away_delta = style_delta[['game_id'] + delta_cols].copy()
# Away team gets negative of home delta
for col in delta_cols:
    away_delta[col] = away_delta[col] * -1
away_delta = away_delta.merge(
    valid_games[['game_id', 'away_team']].rename(columns={'away_team': 'team_name'}),
    on='game_id', how='inner'
)

df_style = pd.concat([home_delta, away_delta], ignore_index=True)

print(f"raw.plays aggregation elapsed : {elapsed:.2f}s")
print(f"Rows in style table           : {len(df_style_raw):,}")
print(f"Games with deltas             : {len(style_delta):,}")
print(f"Rows after expansion          : {len(df_style):,}")
print(f"Expected                      : {len(valid_games) * 2:,}")
print()
for col in delta_cols:
    print(f"  {col:<35s} mean={df_style[col].mean():.4f}  "
          f"null={df_style[col].isna().sum()}")

assert len(df_style) == len(valid_games) * 2, \
    f"Row count mismatch: {len(df_style)} != {len(valid_games) * 2}"
assert elapsed < 60, \
    f"raw.plays aggregation took {elapsed:.2f}s — exceeds 60s limit"

print()
print("Cell 5 complete.")

raw.plays aggregation elapsed : 1.20s
Rows in style table           : 3,214
Games with deltas             : 1,607
Rows after expansion          : 3,214
Expected                      : 3,214

  rush_rate_std_downs_delta           mean=-0.0000  null=0
  rush_rate_pass_downs_delta          mean=-0.0000  null=0
  off_sack_rate_allowed_delta         mean=0.0000  null=0
  def_sack_rate_delta                 mean=-0.0000  null=0

Cell 5 complete.


In [6]:
# =============================================================================
# CELL 6 — Fit KMeans archetypes and build per-team archetype index columns
#
# Exact pipeline from EDA 10. Offense k=4, defense k=4.
# Features aggregated from temp_style_metrics (already built in Cell 5).
#
# OUTPUT per team per game:
#   off_archetype_idx : int, 0-3 — this team's offense archetype
#   def_archetype_idx : int, 0-3 — this team's defense archetype
#
# These are the ONLY archetype columns passed to model_cfb().
# The model indexes into 4-vector embeddings:
#   b_off_archetype[data.off_archetype_idx]
#   b_def_archetype[data.def_archetype_idx]
#
# No compound matchup strings. No 16-way encoding. No matchup_encodings.json.
# Per handoff Critical Issue 1 and model_01/model_02 audit decisions.
#
# Archetype label maps saved to artifacts/archetype_label_maps.json
# for auditability only — not used by the model.
# =============================================================================

# --- Step 1: Build team-game style matrix from temp_style_metrics ---
cur.execute("""
    SELECT
        s.game_id,
        s.season,
        s.team_name,
        s.opponent,
        s.rush_rate_std_downs,
        s.rush_rate_pass_downs,
        s.off_sack_rate_allowed,
        s.def_sack_rate,
        ts.conference
    FROM temp_style_metrics s
    INNER JOIN int.int_team_season_features ts
        ON ts.team_name = s.team_name
       AND ts.season    = s.season
""")
rows = cur.fetchall()
cols = [d[0] for d in cur.description]
team_game_style = pd.DataFrame(rows, columns=cols)

style_numeric = [
    'rush_rate_std_downs', 'rush_rate_pass_downs',
    'off_sack_rate_allowed', 'def_sack_rate'
]
team_game_style[style_numeric] = team_game_style[style_numeric].astype(float)

# --- Step 2: Aggregate to team-season level for clustering ---
# Cluster on season-level tendencies, not game-by-game noise (EDA 10)
offense_features = ['rush_rate_std_downs', 'rush_rate_pass_downs', 'off_sack_rate_allowed']
defense_features = ['def_sack_rate']

team_season_style = (
    team_game_style
    .groupby(['team_name', 'season'])[style_numeric]
    .mean()
    .reset_index()
    .dropna()
)

print(f"Team-season rows for clustering : {len(team_season_style):,}")

# --- Step 3: Fit offense KMeans (k=4) ---
OFFENSE_K    = 4
DEFENSE_K    = 4
RANDOM_STATE = 42

off_matrix = team_season_style[offense_features].values
off_scaler = StandardScaler()
off_scaled = off_scaler.fit_transform(off_matrix)

off_kmeans = KMeans(n_clusters=OFFENSE_K, random_state=RANDOM_STATE, n_init=20)
off_kmeans.fit(off_scaled)
team_season_style['off_archetype_id'] = off_kmeans.labels_

# --- Step 4: Fit defense KMeans (k=4) ---
def_matrix = team_season_style[defense_features].values
def_scaler = StandardScaler()
def_scaled = def_scaler.fit_transform(def_matrix)

def_kmeans = KMeans(n_clusters=DEFENSE_K, random_state=RANDOM_STATE, n_init=20)
def_kmeans.fit(def_scaled)
team_season_style['def_archetype_id'] = def_kmeans.labels_

# --- Step 5: Inspect centroids and assign interpretive labels ---
off_centroids = pd.DataFrame(
    off_scaler.inverse_transform(off_kmeans.cluster_centers_),
    columns=offense_features
)
def_centroids = pd.DataFrame(
    def_scaler.inverse_transform(def_kmeans.cluster_centers_),
    columns=defense_features
)

print("\nOffense cluster centroids:")
print(off_centroids.round(3).to_string())
print("\nDefense cluster centroids:")
print(def_centroids.round(3).to_string())

# Label maps for auditability — NOT used by the model
offense_label_map = {
    0: "high_efficiency_balanced",
    1: "low_efficiency_struggling",
    2: "pass_leaning_efficient",
    3: "run_leaning_limited_pass",
}
defense_label_map = {
    0: "rush_vulnerable_moderate_pass",
    1: "struggling_all_phase",
    2: "strong_all_phase",
    3: "pass_vulnerable_run_stopper",
}

with open(ARTIFACT_DIR / 'archetype_label_maps.json', 'w') as f:
    json.dump({
        'offense_label_map': offense_label_map,
        'defense_label_map': defense_label_map,
    }, f, indent=2)
print("\nSaved: artifacts/archetype_label_maps.json  (auditability only)")

# --- Step 6: Join archetype ids back to game level, expand to two rows ---
# Each team row gets its OWN off_archetype_id and def_archetype_id (0-3).
# This is what model_cfb() indexes into the 4-vector embeddings.
archetype_cols = ['team_name', 'season', 'off_archetype_id', 'def_archetype_id']

game_team_arch = team_game_style[['game_id', 'season', 'team_name']].merge(
    team_season_style[archetype_cols],
    on=['team_name', 'season'],
    how='inner'
)

# Verify: should have exactly two rows per game (one per team)
assert len(game_team_arch) == len(valid_games) * 2, \
    f"Archetype join row count mismatch: {len(game_team_arch)} != {len(valid_games) * 2}"

# Verify: archetype ids are in valid range 0-3
assert game_team_arch['off_archetype_id'].between(0, 3).all(), \
    "off_archetype_id out of range 0-3"
assert game_team_arch['def_archetype_id'].between(0, 3).all(), \
    "def_archetype_id out of range 0-3"

# Verify: no 2025 holdout leaked into clustering
assert HOLDOUT_SEASON not in team_season_style['season'].values, \
    "2025 holdout leaked into archetype clustering"

df_archetypes = game_team_arch.copy()

print(f"\nRows in df_archetypes         : {len(df_archetypes):,}")
print(f"Expected                      : {len(valid_games) * 2:,}")
print(f"\noff_archetype_id distribution:")
print(df_archetypes['off_archetype_id'].value_counts().sort_index().to_string())
print(f"\ndef_archetype_id distribution:")
print(df_archetypes['def_archetype_id'].value_counts().sort_index().to_string())
print()
print("All Cell 6 assertions passed.")
print("Cell 6 complete.")

Team-season rows for clustering : 384

Offense cluster centroids:
   rush_rate_std_downs  rush_rate_pass_downs  off_sack_rate_allowed
0                0.500                 0.279                  0.048
1                0.408                 0.183                  0.049
2                0.450                 0.228                  0.096
3                0.685                 0.435                  0.102

Defense cluster centroids:
   def_sack_rate
0          0.034
1          0.098
2          0.072
3          0.053

Saved: artifacts/archetype_label_maps.json  (auditability only)

Rows in df_archetypes         : 3,214
Expected                      : 3,214

off_archetype_id distribution:
off_archetype_id
0    1282
1    1044
2     805
3      83

def_archetype_id distribution:
def_archetype_id
0     691
1     505
2     909
3    1109

All Cell 6 assertions passed.
Cell 6 complete.


In [7]:
# =============================================================================
# CELL 7 — Assemble all engineered features into a single wide DataFrame
#
# is_home derived by joining to raw.games: team_name = home_team -> 1, else 0.
# is_dome carried from df_env_long (Cell 4).
#
# Null handling applied before standardization:
#   Approach A (impute_season_prior): last3_win_pct, last3_off_epa_avg,
#     last3_def_epa_avg, last3_points_scored_avg, last3_points_allowed_avg,
#     sp_rating, recruiting_3yr_avg
#   Threshold-zeroing: away_elevation_delta_ft (< 2000ft),
#     away_travel_distance_mi (< 1500mi), away_tz_delta_hrs (abs < 2hr),
#     wind_chill (> 40F or is_dome or null), days_since_last_game (< 12 days)
#
# elo_sp_divergence: computed using LOCKED EDA 08 z-score parameters.
#   Do not recompute from sample — must match prediction-time scaling exactly.
#   pregame_elo : mean=1511.6097, std=236.1207
#   sp_rating   : mean=1.0969,    std=12.8712
#
# Standardization rules (per handoff preprocessing spec):
#   Continuous features       : mean=0, std=1 (fit on training data)
#   Sparse threshold-activated: winsorize to physical cap THEN divide by
#                               std of non-zero values only, no mean subtraction
#                               — zeros must stay zero
#   elo_sp_divergence         : already a z-score difference — do not re-scale
#   is_home                   : binary — no standardization
#   off_archetype_idx,
#   def_archetype_idx         : categorical int — no standardization
#
# Winsorization caps (applied to non-zero sparse features before scaling,
# and to continuous features after standardization):
#   days_since_last_game      : cap at 21 days     (max reasonable bye gap)
#   away_travel_distance_mi   : cap at 5000 mi
#   away_elevation_delta_ft   : cap at 7000 ft
#   wind_chill                : cap at -20 F       (floor — it's cold)
#   Continuous after std      : clip to [-3, 3] sigma
#
# scaler_stats.json written after standardization — required for prediction.
# =============================================================================

# --- Step 1: Load base features ---
base_sql = """
    SELECT
        f.game_id,
        f.season,
        f.team_name,
        f.opponent,
        CASE WHEN f.team_name = g.home_team THEN 1 ELSE 0 END AS is_home,
        f.points_scored,
        f.close_game_epa_per_play,
        f.close_game_def_epa_per_play,
        f.pregame_elo,
        f.opponent_pregame_elo,
        f.last3_win_pct,
        f.last3_off_epa_avg,
        f.last3_def_epa_avg,
        f.last3_points_scored_avg,
        f.last3_points_allowed_avg,
        f.days_since_last_game,
        ts.conference,
        ts.sp_rating,
        ts.recruiting_3yr_avg
    FROM int.int_game_team_features f
    INNER JOIN temp_valid_fbs_games vg
        ON vg.game_id = f.game_id
    INNER JOIN raw.games g
        ON g.id     = f.game_id
       AND g.season = f.season
    INNER JOIN int.int_team_season_features ts
        ON ts.team_name = f.team_name
       AND ts.season    = f.season
    WHERE f.season IN (2022, 2023, 2024)
"""

cur.execute(base_sql)
rows = cur.fetchall()
cols = [d[0] for d in cur.description]
df = pd.DataFrame(rows, columns=cols)

base_numeric = [
    'points_scored', 'is_home',
    'close_game_epa_per_play', 'close_game_def_epa_per_play',
    'pregame_elo', 'opponent_pregame_elo',
    'last3_win_pct', 'last3_off_epa_avg', 'last3_def_epa_avg',
    'last3_points_scored_avg', 'last3_points_allowed_avg',
    'days_since_last_game', 'sp_rating', 'recruiting_3yr_avg',
]
df[base_numeric] = df[base_numeric].astype(float)

print(f"Base rows loaded : {len(df):,}")

# --- Step 2: Merge engineered feature sets ---
df = df.merge(
    df_close[['game_id', 'team_name', 'close_game_play_count_delta']],
    on=['game_id', 'team_name'], how='left'
)
df = df.merge(
    df_env_long[['game_id', 'team_name', 'wind_chill', 'is_dome',
                 'away_elevation_delta_ft', 'away_travel_distance_mi',
                 'away_tz_delta_hrs']],
    on=['game_id', 'team_name'], how='left'
)
df = df.merge(
    df_style[['game_id', 'team_name',
              'rush_rate_std_downs_delta', 'rush_rate_pass_downs_delta',
              'off_sack_rate_allowed_delta', 'def_sack_rate_delta']],
    on=['game_id', 'team_name'], how='left'
)
df = df.merge(
    df_archetypes[['game_id', 'team_name', 'off_archetype_id', 'def_archetype_id']],
    on=['game_id', 'team_name'], how='left'
)

print(f"Rows after merges : {len(df):,}")
assert len(df) == len(valid_games) * 2, \
    f"Row count mismatch after merges: {len(df)}"

# --- Step 3: Compute elo_sp_divergence using locked EDA 08 parameters ---
ELO_MEAN = 1511.6097
ELO_STD  = 236.1207
SP_MEAN  =    1.0969
SP_STD   =   12.8712

df['elo_sp_divergence'] = (
    (df['pregame_elo'] - ELO_MEAN) / ELO_STD
) - (
    (df['sp_rating']  - SP_MEAN)  / SP_STD
)

print(f"\nelo_sp_divergence mean : {df['elo_sp_divergence'].mean():.4f}  (expect ≈ 0.000)")
print(f"elo_sp_divergence std  : {df['elo_sp_divergence'].std():.4f}   (expect ≈ 0.45)")

# --- Step 4: Approach A — impute_season_prior ---
impute_cols = [
    'last3_win_pct', 'last3_off_epa_avg', 'last3_def_epa_avg',
    'last3_points_scored_avg', 'last3_points_allowed_avg',
    'sp_rating', 'recruiting_3yr_avg',
]

df = df.sort_values(['season', 'game_id']).reset_index(drop=True)

for col in impute_cols:
    season_prior = (
        df.groupby(['team_name', 'season'])[col]
        .transform(lambda x: x.expanding().mean().shift(1))
    )
    season_mean = df.groupby(['team_name', 'season'])[col].transform('mean')
    season_prior = season_prior.fillna(season_mean)
    league_mean  = df[col].mean()
    df[col] = df[col].fillna(season_prior).fillna(league_mean)

print(f"\nNulls after Approach A imputation:")
print(df[impute_cols].isna().sum().to_string())

# --- Step 5: Threshold-zeroing ---
df['away_elevation_delta_ft'] = np.where(
    df['away_elevation_delta_ft'] >= 2000,
    df['away_elevation_delta_ft'], 0.0
)
df['away_travel_distance_mi'] = np.where(
    df['away_travel_distance_mi'] >= 1500,
    df['away_travel_distance_mi'], 0.0
)
df['away_tz_delta_hrs'] = np.where(
    df['away_tz_delta_hrs'].abs() >= 2,
    df['away_tz_delta_hrs'], 0.0
)
df['wind_chill'] = np.where(
    (df['wind_chill'] <= 40) & (df['is_dome'] == 0) & df['wind_chill'].notna(),
    df['wind_chill'], 0.0
)
df['days_since_last_game'] = np.where(
    df['days_since_last_game'] >= 12,
    df['days_since_last_game'], 0.0
)
df['close_game_epa_per_play']     = df['close_game_epa_per_play'].fillna(0.0)
df['close_game_def_epa_per_play'] = df['close_game_def_epa_per_play'].fillna(0.0)

print(f"\nThreshold-zeroing results:")
print(f"  away_elevation non-zero : {(df['away_elevation_delta_ft'] != 0).sum():,}")
print(f"  away_travel non-zero    : {(df['away_travel_distance_mi'] != 0).sum():,}")
print(f"  away_tz non-zero        : {(df['away_tz_delta_hrs'].abs() > 0).sum():,}")
print(f"  wind_chill non-zero     : {(df['wind_chill'] != 0).sum():,}")
print(f"  days_since non-zero     : {(df['days_since_last_game'] != 0).sum():,}")

# --- Step 6: Winsorization ---
# Applied AFTER threshold-zeroing, BEFORE standardization.
# Caps extreme physical values to prevent scaling artifacts.
# Caps are applied to non-zero values only for sparse features.
# Continuous features are clipped to [-3, 3] after standardization (step 7b).

SPARSE_CAPS = {
    'days_since_last_game'    : 21.0,    # max reasonable bye gap in days
    'away_travel_distance_mi' : 5000.0,  # Hawaii is ~5100mi — just under cap
    'away_elevation_delta_ft' : 7000.0,  # highest CFB venue ~7200ft (Air Force)
    'wind_chill'              : None,    # no cap — already threshold-zeroed at 40F;
                                         # cold end bounded by physical reality
}

print(f"\nWinsorization (sparse features, non-zero values only):")
for col, cap in SPARSE_CAPS.items():
    if cap is None:
        print(f"  {col:<35}: no cap applied")
        continue
    n_before = (df[col] > cap).sum()
    df[col] = np.where(
        (df[col] != 0) & (df[col] > cap), cap,
        np.where(
            (df[col] != 0) & (df[col] < -cap), -cap,
            df[col]
        )
    )
    n_after = (df[col] > cap).sum()
    print(f"  {col:<35}: cap=±{cap:.0f}  rows capped: {n_before - n_after}")

print(f"\nPost-winsorization sparse feature ranges (non-zero):")
for col in SPARSE_CAPS:
    nonzero = df.loc[df[col] != 0, col]
    if len(nonzero) > 0:
        print(f"  {col:<35}: min={nonzero.min():.1f}  max={nonzero.max():.1f}  "
              f"n={len(nonzero)}")

# --- Step 7: Standardization ---
scaler_stats = {}

# 7a. Continuous features — mean=0, std=1, then clip to [-3, 3]
CONTINUOUS_FEATURES = [
    'close_game_epa_per_play',
    'close_game_def_epa_per_play',
    'pregame_elo',
    'last3_win_pct',
    'last3_off_epa_avg',
    'last3_def_epa_avg',
    'last3_points_scored_avg',
    'last3_points_allowed_avg',
    'sp_rating',
    'recruiting_3yr_avg',
    'rush_rate_std_downs_delta',
    'rush_rate_pass_downs_delta',
    'off_sack_rate_allowed_delta',
    'def_sack_rate_delta',
    'close_game_play_count_delta',
]

for col in CONTINUOUS_FEATURES:
    mean = df[col].mean()
    std  = df[col].std()
    df[col] = ((df[col] - mean) / std).clip(-3.0, 3.0)
    scaler_stats[col] = {
        'mean': round(float(mean), 6),
        'std':  round(float(std),  6),
        'clip': '[-3, 3]',
    }

# 7b. Sparse threshold-activated — divide by non-zero std only, no mean subtraction
# Winsorization already applied in Step 6, so non-zero std is now well-behaved.
SPARSE_FEATURES = [
    'away_elevation_delta_ft',
    'away_travel_distance_mi',
    'away_tz_delta_hrs',
    'wind_chill',
    'days_since_last_game',
]

for col in SPARSE_FEATURES:
    nonzero_vals = df.loc[df[col] != 0, col]
    std_nonzero  = nonzero_vals.std()
    df[col] = np.where(df[col] != 0, df[col] / std_nonzero, 0.0)
    scaler_stats[col] = {
        'mean':       0.0,
        'std':        round(float(std_nonzero), 6),
        'n_nonzero':  int(len(nonzero_vals)),
        'method':     'sparse_nonzero_std',
    }

# 7c. elo_sp_divergence — already a z-score difference, do not re-scale
scaler_stats['elo_sp_divergence'] = {
    'mean':    0.0,
    'std':     None,
    'method':  'passthrough_zscore_difference',
    'elo_mean': ELO_MEAN,
    'elo_std':  ELO_STD,
    'sp_mean':  SP_MEAN,
    'sp_std':   SP_STD,
}

print(f"\nStandardization complete.")
print(f"  Continuous features scaled    : {len(CONTINUOUS_FEATURES)}")
print(f"  Sparse features scaled        : {len(SPARSE_FEATURES)}")
print(f"  Not scaled (passthrough)      : elo_sp_divergence, is_home, archetypes")

# --- Step 8: Write scaler_stats.json ---
with open(ARTIFACT_DIR / 'scaler_stats.json', 'w') as f:
    json.dump(scaler_stats, f, indent=2)
print(f"\nSaved: artifacts/scaler_stats.json  ({len(scaler_stats)} features)")

# --- Step 9: Post-scaling range validation ---
print(f"\nPost-scaling feature range check (all should be |max| <= 3.0):")
check_cols = CONTINUOUS_FEATURES + SPARSE_FEATURES
all_ok = True
for col in check_cols:
    abs_max = df[col].abs().max()
    flag = "" if abs_max <= 3.05 else " ← EXCEEDS 3 SIGMA"
    print(f"  {col:<35}: abs_max={abs_max:.3f}{flag}")
    if abs_max > 3.05:
        all_ok = False

if all_ok:
    print("\nAll features within bounds ✓")
else:
    print("\nWARNING: some features exceed 3 sigma — review before proceeding")

# --- Step 10: Final null check ---
final_cols = [
    'points_scored', 'is_home', 'conference', 'sp_rating', 'recruiting_3yr_avg',
    'close_game_epa_per_play', 'close_game_def_epa_per_play',
    'pregame_elo', 'elo_sp_divergence',
    'last3_win_pct', 'away_travel_distance_mi', 'away_tz_delta_hrs',
    'wind_chill', 'away_elevation_delta_ft',
    'rush_rate_std_downs_delta', 'rush_rate_pass_downs_delta',
    'off_sack_rate_allowed_delta', 'def_sack_rate_delta',
    'off_archetype_id', 'def_archetype_id',
    'close_game_play_count_delta', 'days_since_last_game',
    'last3_off_epa_avg', 'last3_def_epa_avg',
    'last3_points_scored_avg', 'last3_points_allowed_avg',
]

null_counts = df[final_cols].isna().sum()
if null_counts.sum() > 0:
    print(f"\nNulls remaining in final columns:")
    print(null_counts[null_counts > 0].to_string())
else:
    print(f"\nNo nulls remaining in any model feature column.")

assert null_counts.sum() == 0, \
    "Nulls remain in model features — fix before writing to DB"

print(f"\nFinal DataFrame shape : {df.shape}")
print()
print("Cell 7 complete.")

Base rows loaded : 3,214
Rows after merges : 3,214

elo_sp_divergence mean : -0.0001  (expect ≈ 0.000)
elo_sp_divergence std  : 0.4487   (expect ≈ 0.45)

Nulls after Approach A imputation:
last3_win_pct               0
last3_off_epa_avg           0
last3_def_epa_avg           0
last3_points_scored_avg     0
last3_points_allowed_avg    0
sp_rating                   0
recruiting_3yr_avg          0

Threshold-zeroing results:
  away_elevation non-zero : 127
  away_travel non-zero    : 82
  away_tz non-zero        : 85
  wind_chill non-zero     : 344
  days_since non-zero     : 443

Winsorization (sparse features, non-zero values only):
  days_since_last_game               : cap=±21  rows capped: 0
  away_travel_distance_mi            : cap=±5000  rows capped: 0
  away_elevation_delta_ft            : cap=±7000  rows capped: 3
  wind_chill                         : no cap applied

Post-winsorization sparse feature ranges (non-zero):
  days_since_last_game               : min=12.0  max=18.0 

In [8]:
# =============================================================================
# CELL 8 — Write to int.int_game_model_features
# Uses psycopg2.extras.execute_values for fast bulk insert (single round trip)
# =============================================================================
from psycopg2.extras import execute_values

# --- Step 1: Create table ---
create_sql = """
    DROP TABLE IF EXISTS int.int_game_model_features;

    CREATE TABLE int.int_game_model_features (
        game_id                         INTEGER      NOT NULL,
        season                          INTEGER      NOT NULL,
        team_name                       TEXT         NOT NULL,
        opponent                        TEXT         NOT NULL,
        conference                      TEXT         NOT NULL,
        is_home                         SMALLINT     NOT NULL,
        points_scored                   NUMERIC      NOT NULL,
        sp_rating                       NUMERIC      NOT NULL,
        recruiting_3yr_avg              NUMERIC      NOT NULL,
        close_game_epa_per_play         NUMERIC      NOT NULL,
        close_game_def_epa_per_play     NUMERIC      NOT NULL,
        pregame_elo                     NUMERIC      NOT NULL,
        opponent_pregame_elo            NUMERIC      NOT NULL,
        elo_sp_divergence               NUMERIC      NOT NULL,
        last3_win_pct                   NUMERIC      NOT NULL,
        away_travel_distance_mi         NUMERIC      NOT NULL,
        away_tz_delta_hrs               NUMERIC      NOT NULL,
        wind_chill                      NUMERIC      NOT NULL,
        rush_rate_std_downs_delta       NUMERIC      NOT NULL,
        rush_rate_pass_downs_delta      NUMERIC      NOT NULL,
        off_sack_rate_allowed_delta     NUMERIC      NOT NULL,
        def_sack_rate_delta             NUMERIC      NOT NULL,
        off_archetype_idx               SMALLINT     NOT NULL,
        def_archetype_idx               SMALLINT     NOT NULL,
        last3_off_epa_avg               NUMERIC      NOT NULL,
        last3_def_epa_avg               NUMERIC      NOT NULL,
        last3_points_scored_avg         NUMERIC      NOT NULL,
        last3_points_allowed_avg        NUMERIC      NOT NULL,
        days_since_last_game            NUMERIC      NOT NULL,
        close_game_play_count_delta     NUMERIC      NOT NULL,
        away_elevation_delta_ft         NUMERIC      NOT NULL,
        PRIMARY KEY (game_id, team_name)
    );
"""

cur.execute(create_sql)
conn.commit()
print("Table created.")

# --- Step 2: Build insert list using numpy — no iterrows() ---
insert_cols = [
    'game_id', 'season', 'team_name', 'opponent', 'conference', 'is_home',
    'points_scored', 'sp_rating', 'recruiting_3yr_avg',
    'close_game_epa_per_play', 'close_game_def_epa_per_play',
    'pregame_elo', 'opponent_pregame_elo', 'elo_sp_divergence',
    'last3_win_pct', 'away_travel_distance_mi', 'away_tz_delta_hrs',
    'wind_chill',
    'rush_rate_std_downs_delta', 'rush_rate_pass_downs_delta',
    'off_sack_rate_allowed_delta', 'def_sack_rate_delta',
    'off_archetype_id', 'def_archetype_id',
    'last3_off_epa_avg', 'last3_def_epa_avg',
    'last3_points_scored_avg', 'last3_points_allowed_avg',
    'days_since_last_game', 'close_game_play_count_delta',
    'away_elevation_delta_ft',
]

db_cols = ', '.join(
    c.replace('off_archetype_id', 'off_archetype_idx')
     .replace('def_archetype_id', 'def_archetype_idx')
    for c in insert_cols
)

# Convert to list of tuples via df.values — orders of magnitude faster than iterrows
rows_to_insert = [
    tuple(row) for row in df[insert_cols].itertuples(index=False, name=None)
]

insert_sql = f"INSERT INTO int.int_game_model_features ({db_cols}) VALUES %s"

execute_values(cur, insert_sql, rows_to_insert, page_size=500)
conn.commit()

print(f"Rows inserted : {len(rows_to_insert):,}")

# --- Step 3: Verify ---
cur.execute("SELECT COUNT(*) FROM int.int_game_model_features")
db_count = cur.fetchone()[0]
print(f"Rows in DB    : {db_count:,}")
assert db_count == len(valid_games) * 2, \
    f"DB row count {db_count} != expected {len(valid_games) * 2}"

cur.execute("""
    SELECT column_name FROM information_schema.columns
    WHERE table_schema = 'int' AND table_name = 'int_game_model_features'
    ORDER BY ordinal_position
""")
db_cols_list = [row[0] for row in cur.fetchall()]
print(f"Columns in DB : {len(db_cols_list)}")
for col in db_cols_list:
    print(f"  {col}")

assert 'off_archetype_idx' in db_cols_list
assert 'def_archetype_idx' in db_cols_list
stale = {'offense_archetype_matchup', 'defense_archetype_matchup',
         'home_off_vs_away_def_matchup', 'away_off_vs_home_def_matchup'}
assert not stale & set(db_cols_list), f"Stale columns found: {stale & set(db_cols_list)}"

print("\nAll assertions passed.")
print("Cell 8 complete.")

Table created.
Rows inserted : 3,214
Rows in DB    : 3,214
Columns in DB : 31
  game_id
  season
  team_name
  opponent
  conference
  is_home
  points_scored
  sp_rating
  recruiting_3yr_avg
  close_game_epa_per_play
  close_game_def_epa_per_play
  pregame_elo
  opponent_pregame_elo
  elo_sp_divergence
  last3_win_pct
  away_travel_distance_mi
  away_tz_delta_hrs
  wind_chill
  rush_rate_std_downs_delta
  rush_rate_pass_downs_delta
  off_sack_rate_allowed_delta
  def_sack_rate_delta
  off_archetype_idx
  def_archetype_idx
  last3_off_epa_avg
  last3_def_epa_avg
  last3_points_scored_avg
  last3_points_allowed_avg
  days_since_last_game
  close_game_play_count_delta
  away_elevation_delta_ft

All assertions passed.
Cell 8 complete.


In [9]:
# =============================================================================
# CELL 9 — Validation
#
# Reads back from int.int_game_model_features and verifies:
#   - Row count matches training game pool x 2
#   - No nulls in any column
#   - FBS Independents absent
#   - Season 2025 absent
#   - is_home distribution is balanced (exactly 50/50)
#   - points_scored range is plausible (0-100)
#   - Conference distribution matches Cell 2
#   - Threshold-activated features have correct zero proportions
#   - off_archetype_idx and def_archetype_idx values are in range 0-3
#   - No compound matchup string columns present
# =============================================================================

cur.execute("SELECT * FROM int.int_game_model_features")
rows = cur.fetchall()
cols = [d[0] for d in cur.description]
df_val = pd.DataFrame(rows, columns=cols)

numeric_val_cols = [c for c in df_val.columns if c not in
                    ['team_name', 'opponent', 'conference']]
df_val[numeric_val_cols] = df_val[numeric_val_cols].astype(float)

print("=" * 60)
print("VALIDATION — int.int_game_model_features")
print("=" * 60)

# Row count
print(f"\nRow count          : {len(df_val):,}")
print(f"Expected           : {len(valid_games) * 2:,}")
assert len(df_val) == len(valid_games) * 2, "Row count mismatch"

# Seasons
seasons = sorted(df_val['season'].unique().tolist())
print(f"Seasons            : {seasons}")
assert HOLDOUT_SEASON not in df_val['season'].values, \
    "2025 holdout found in int_game_model_features"
assert set(seasons) == {2022, 2023, 2024}, \
    f"Unexpected seasons: {seasons}"

# No nulls
null_counts = df_val.isna().sum()
if null_counts.sum() > 0:
    print("\nNulls found:")
    print(null_counts[null_counts > 0].to_string())
    assert False, "Nulls found in int_game_model_features"
else:
    print("Nulls              : none")

# FBS Independents absent
assert 'FBS Independents' not in df_val['conference'].values, \
    "FBS Independents found in conference column"
print("FBS Independents   : absent ✓")

# is_home balance — must be exactly 50/50
home_count = int((df_val['is_home'] == 1).sum())
away_count = int((df_val['is_home'] == 0).sum())
print(f"is_home=1          : {home_count:,}")
print(f"is_home=0          : {away_count:,}")
assert home_count == away_count == len(valid_games), \
    f"is_home imbalance: {home_count} home, {away_count} away"

# points_scored range
pts_min  = df_val['points_scored'].min()
pts_max  = df_val['points_scored'].max()
pts_mean = df_val['points_scored'].mean()
print(f"points_scored      : min={pts_min:.0f}  max={pts_max:.0f}  mean={pts_mean:.1f}")
assert pts_min >= 0,   "Negative points_scored found"
assert pts_max <= 100, f"Implausible points_scored max: {pts_max}"

# Conference distribution
print(f"\nConference distribution:")
print(df_val['conference'].value_counts().sort_index().to_string())

# Threshold feature zero proportions
print(f"\nThreshold-activated feature non-zero counts:")
print(f"  away_elevation_delta_ft  != 0 : "
      f"{(df_val['away_elevation_delta_ft'] != 0).sum():,}")
print(f"  away_travel_distance_mi  != 0 : "
      f"{(df_val['away_travel_distance_mi'] != 0).sum():,}")
print(f"  away_tz_delta_hrs        != 0 : "
      f"{(df_val['away_tz_delta_hrs'] != 0).sum():,}")
print(f"  wind_chill               != 0 : "
      f"{(df_val['wind_chill'] != 0).sum():,}")
print(f"  days_since_last_game     != 0 : "
      f"{(df_val['days_since_last_game'] != 0).sum():,}")

# Archetype index validation
print(f"\nArchetype index validation:")
for col in ['off_archetype_idx', 'def_archetype_idx']:
    vals = df_val[col].astype(int)
    print(f"  {col} range : {vals.min()}-{vals.max()}  "
          f"distribution: {vals.value_counts().sort_index().to_dict()}")
    assert vals.between(0, 3).all(), \
        f"{col} contains values outside 0-3"

# No compound matchup columns
stale_cols = {'offense_archetype_matchup', 'defense_archetype_matchup',
              'home_off_vs_away_def_matchup', 'away_off_vs_home_def_matchup'}
found_stale = stale_cols & set(df_val.columns)
assert not found_stale, f"Stale compound matchup columns found: {found_stale}"
print("  Compound matchup columns   : absent ✓")

# Unique teams and games
print(f"\nUnique teams       : {df_val['team_name'].nunique():,}")
print(f"Unique games       : {df_val['game_id'].nunique():,}")

print()
print("All validation checks passed.")
print()
print("Cell 9 complete.")

VALIDATION — int.int_game_model_features

Row count          : 3,214
Expected           : 3,214
Seasons            : [2022.0, 2023.0, 2024.0]
Nulls              : none
FBS Independents   : absent ✓
is_home=1          : 1,607
is_home=0          : 1,607
points_scored      : min=0  max=77  mean=26.9

Conference distribution:
conference
ACC                  364
American Athletic    320
Big 12               366
Big Ten              420
Conference USA       246
Mid-American         294
Mountain West        282
Pac-12               222
SEC                  358
Sun Belt             342

Threshold-activated feature non-zero counts:
  away_elevation_delta_ft  != 0 : 127
  away_travel_distance_mi  != 0 : 82
  away_tz_delta_hrs        != 0 : 85
  wind_chill               != 0 : 344
  days_since_last_game     != 0 : 443

Archetype index validation:
  off_archetype_idx range : 0-3  distribution: {0: 1282, 1: 1044, 2: 805, 3: 83}
  def_archetype_idx range : 0-3  distribution: {0: 691, 1: 505, 2: 909,

## Notebook Summary — model_03_feature_engineering.ipynb

### What This Notebook Does

Builds all engineered features required by `model_cfb()` and writes them to
`int.int_game_model_features` — one row per team per game, covering the
2022–2024 training seasons. Every subsequent model notebook reads from this
table. No feature engineering occurs downstream of this notebook.

### Why This Notebook Exists

Several features included in `artifacts/final_features.csv` were computed
in-memory during EDA and never persisted to the database:

- `wind_chill` — computed in EDA 06, not saved
- `rush_rate_std_downs_delta`, `rush_rate_pass_downs_delta`, `off_sack_rate_allowed_delta`, `def_sack_rate_delta` — computed in EDA 09
- `off_archetype_idx`, `def_archetype_idx` — KMeans archetype ids computed in EDA 10; stored here as per-team integer indices (0–3), not compound matchup strings
- `close_game_play_count_delta` — derivable from existing columns but not stored
- `is_home` — not in `int_game_team_features`; derived from `raw.games`
- `elo_sp_divergence` — z-score difference computed using locked EDA 08 parameters

### Cell-by-Cell Summary

| Cell | Purpose | Output |
|------|---------|--------|
| 1 | Imports, environment, DB connection | `conn`, `cur` open |
| 2 | Valid FBS game pool | `temp_valid_fbs_games` (1,607 games) |
| 3 | `close_game_play_count_delta` | `df_close` (3,214 rows) |
| 4 | `wind_chill` + environmental columns | `df_env_long` (3,214 rows) |
| 5 | Rush/sack rate deltas from `raw.plays` | `df_style` (3,214 rows) |
| 6 | KMeans archetypes (offense k=4, defense k=4) | `df_archetypes` (3,214 rows) |
| 7 | Assemble wide DataFrame, null handling, standardization, `scaler_stats.json` | `df` (3,214 × 32) |
| 8 | Write to `int.int_game_model_features` | 3,214 rows, 31 columns |
| 9 | Validation | all checks passed |

### Null Handling

- **Approach A (impute_season_prior):** `last3_win_pct`, `last3_off_epa_avg`, `last3_def_epa_avg`, `last3_points_scored_avg`, `last3_points_allowed_avg`, `sp_rating`, `recruiting_3yr_avg` — expanding season-to-date mean, fallback to full season mean, fallback to league mean.
- **Threshold-zeroing:** `away_elevation_delta_ft` (< 2000ft), `away_travel_distance_mi` (< 1500mi), `away_tz_delta_hrs` (abs < 2hr), `wind_chill` (> 40F or dome or null), `days_since_last_game` (< 12 days), `close_game_epa_per_play` / `close_game_def_epa_per_play` (null = no close-game situations → zero).

### Standardization

Applied in Cell 7 after null handling and threshold-zeroing. Rules per handoff preprocessing spec:

- **Continuous features** (15): standardized to mean=0, std=1. Parameters saved to `artifacts/scaler_stats.json`.
- **Sparse threshold-activated features** (5): divided by std of non-zero values only — no mean subtraction. Zeros remain zero. Parameters saved to `artifacts/scaler_stats.json`.
- **`elo_sp_divergence`**: already a z-score difference using locked EDA 08 parameters (`pregame_elo` mean=1511.6097, std=236.1207; `sp_rating` mean=1.0969, std=12.8712). Not re-standardized.
- **`is_home`**: binary — no standardization.
- **`off_archetype_idx`, `def_archetype_idx`**: categorical integer — no standardization.

### Archetype Encoding

KMeans refit on every run (offense k=4, defense k=4, random_state=42) on team-season aggregated style metrics. Each team-game row receives its own `off_archetype_idx` (0–3) and `def_archetype_idx` (0–3). These are indexed into 4-vector embeddings in `model_cfb()`:

```python
b_off_archetype[data.off_archetype_idx]
b_def_archetype[data.def_archetype_idx]
```

No compound matchup string columns are built or stored. The 16-way matchup encoding used in earlier EDA versions is not carried forward. Label maps saved to `artifacts/archetype_label_maps.json` for auditability only.

### Artifacts Written

| Artifact | Contents |
|----------|----------|
| `artifacts/scaler_stats.json` | Mean, std (or non-zero std) for all 21 scaled features |
| `artifacts/archetype_label_maps.json` | Interpretive cluster labels for offense and defense archetypes |

### Key Facts

- 1,607 FBS conference games, seasons 2022–2024
- 131 unique teams
- mean `points_scored` = 26.9 (matches FBS baseline)
- 2025 holdout: never touched
- FBS Independents: excluded from all rows

### What the Next Notebook Does

`model_04_first_fit.ipynb` loads from `int.int_game_model_features`, builds index arrays, casts `off_archetype_idx` and `def_archetype_idx` to `int32`, constructs `GameData`, and passes `model_cfb()` to `numpyro.infer.MCMC` with NUTS for the first fit on 2022–2024 training data.